In [ ]:
# Data Graph Explorer
# This notebook allows you to explore CSV data through interactive graphs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO
import requests
from google.colab import files

print("=== Data Graph Explorer ===")
print("Choose how to load your CSV file:")

def load_csv():
    """
    Load CSV data from different sources
    Returns: pandas DataFrame
    """
    print("\n1. Upload from local computer")
    print("2. Enter URL manually")
    print("3. Use predefined URL")

    choice = input("\nEnter your choice (1, 2, or 3): ").strip()

    if choice == "1":
        # Option 1: Upload from local computer
        print("\nPlease upload your CSV file:")
        uploaded = files.upload()

        if uploaded:
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(filename)
            print(f"✅ Successfully loaded '{filename}'")
            return df
        else:
            print("❌ No file uploaded. Using sample data instead.")
            return load_sample_data()

    elif choice == "2":
        # Option 2: Get URL from user input
        url = input("\nEnter the CSV file URL: ").strip()
        try:
            response = requests.get(url)
            response.raise_for_status()
            df = pd.read_csv(StringIO(response.text))
            print("✅ Successfully loaded data from URL")
            return df
        except Exception as e:
            print(f"❌ Error loading from URL: {e}")
            print("Using sample data instead.")
            return load_sample_data()

    elif choice == "3":
        # Option 3: Predefined URLs
        print("\nAvailable sample datasets:")
        sample_urls = {
            "1": "https://raw.githubusercontent.com/datasets/iris/master/data/iris.csv",
            "2": "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv",
            "3": "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
        }

        for key, url in sample_urls.items():
            print(f"{key}. {url.split('/')[-2]}")

        sample_choice = input("\nChoose a sample dataset (1, 2, or 3): ").strip()
        url = sample_urls.get(sample_choice, sample_urls["1"])

        try:
            response = requests.get(url)
            response.raise_for_status()
            df = pd.read_csv(StringIO(response.text))
            print("✅ Successfully loaded sample data")
            return df
        except Exception as e:
            print(f"❌ Error loading sample data: {e}")
            return load_sample_data()

    else:
        print("❌ Invalid choice. Using sample data.")
        return load_sample_data()

def load_sample_data():
    """Load sample data if other methods fail"""
    print("\nLoading built-in sample data...")
    # Create sample data
    data = {
        'Age': [25, 30, 35, 40, 45, 50, 55, 60, 65, 70],
        'Salary': [50000, 60000, 75000, 80000, 90000, 95000, 100000, 110000, 115000, 120000],
        'Experience': [2, 5, 8, 12, 15, 18, 20, 25, 30, 35],
        'Performance': [75, 82, 88, 85, 90, 92, 88, 95, 90, 85]
    }
    return pd.DataFrame(data)

def display_data_info(df):
    """Display dataframe information and sample rows"""
    print("\n" + "="*50)
    print("📊 DATASET INFORMATION")
    print("="*50)

    # Basic info
    print(f"Dataset shape: {df.shape}")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")

    # Column names as list
    columns_list = df.columns.tolist()
    print(f"\n📋 Column names: {columns_list}")

    # Data types
    print("\n📝 Data types:")
    print(df.dtypes)

    # Headings and first two rows
    print("\n📍 First two rows:")
    print(df.head(2))

    # Basic statistics
    print("\n📈 Basic statistics:")
    print(df.describe())

    return columns_list

def create_graph(df, columns_list):
    """Create scatter or line graph based on user selection"""
    print("\n" + "="*50)
    print("📊 GRAPH CREATION")
    print("="*50)

    # Display available columns
    print("Available columns:")
    for i, col in enumerate(columns_list, 1):
        print(f"{i}. {col}")

    # Column selection
    try:
        if len(columns_list) >= 2:
            print("\nChoose 1 column for line graph, or 2 columns for scatter plot")
            choices = input("Enter column numbers (e.g., '1' or '1 2'): ").strip().split()

            if len(choices) == 1:
                # Single column - line graph
                col_idx = int(choices[0]) - 1
                if 0 <= col_idx < len(columns_list):
                    column = columns_list[col_idx]
                    create_line_graph(df, column)
                else:
                    print("❌ Invalid column selection")

            elif len(choices) == 2:
                # Two columns - scatter plot
                col1_idx = int(choices[0]) - 1
                col2_idx = int(choices[1]) - 1

                if (0 <= col1_idx < len(columns_list) and
                    0 <= col2_idx < len(columns_list)):
                    col1 = columns_list[col1_idx]
                    col2 = columns_list[col2_idx]
                    create_scatter_plot(df, col1, col2)
                else:
                    print("❌ Invalid column selection")

            else:
                print("❌ Please select 1 or 2 columns")

        else:
            print("❌ Not enough columns for graphing")

    except ValueError:
        print("❌ Please enter valid numbers")
    except Exception as e:
        print(f"❌ Error creating graph: {e}")

def create_scatter_plot(df, col1, col2):
    """Create a scatter plot from two columns"""
    # Convert to numpy arrays
    x_data = df[col1].to_numpy()
    y_data = df[col2].to_numpy()

    print(f"\n📊 Creating scatter plot: {col1} vs {col2}")
    print(f"X data (first 5 values): {x_data[:5]}")
    print(f"Y data (first 5 values): {y_data[:5]}")

    # Create plot
    plt.figure(figsize=(10, 6))
    plt.scatter(x_data, y_data, alpha=0.7, color='blue', s=50)
    plt.xlabel(col1)
    plt.ylabel(col2)
    plt.title(f'Scatter Plot: {col1} vs {col2}')
    plt.grid(True, alpha=0.3)

    # Add correlation coefficient
    correlation = np.corrcoef(x_data, y_data)[0, 1]
    plt.text(0.05, 0.95, f'Correlation: {correlation:.2f}',
             transform=plt.gca().transAxes, bbox=dict(boxstyle="round", facecolor='wheat'))

    plt.tight_layout()
    plt.show()

    # Interpretation
    print(f"\n🔍 INTERPRETATION:")
    print(f"Scatter plot shows relationship between '{col1}' and '{col2}'")
    print(f"Correlation coefficient: {correlation:.2f}")

    if abs(correlation) > 0.7:
        strength = "strong"
    elif abs(correlation) > 0.3:
        strength = "moderate"
    else:
        strength = "weak"

    if correlation > 0:
        direction = "positive"
    else:
        direction = "negative"

    print(f"This indicates a {strength} {direction} relationship")
    print("• Points clustered together suggest consistent relationship")
    print("• Widely scattered points suggest weak relationship")
    print("• Upward trend indicates positive correlation")
    print("• Downward trend indicates negative correlation")

def create_line_graph(df, column):
    """Create a line graph from a single column"""
    # Convert to numpy array
    y_data = df[column].to_numpy()
    x_data = np.arange(len(y_data))  # Use index as x-axis

    print(f"\n📈 Creating line graph for: {column}")
    print(f"Data (first 10 values): {y_data[:10]}")

    # Create plot
    plt.figure(figsize=(12, 6))
    plt.plot(x_data, y_data, marker='o', linewidth=2, markersize=4)
    plt.xlabel('Index')
    plt.ylabel(column)
    plt.title(f'Line Graph: {column} over Index')
    plt.grid(True, alpha=0.3)

    # Add statistics
    mean_val = np.mean(y_data)
    plt.axhline(y=mean_val, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_val:.2f}')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Interpretation
    print(f"\n🔍 INTERPRETATION:")
    print(f"Line graph shows trend of '{column}' across the dataset")
    print(f"Mean value: {mean_val:.2f}")
    print(f"Data range: {np.min(y_data):.2f} to {np.max(y_data):.2f}")

    # Trend analysis
    if len(y_data) > 1:
        trend = "increasing" if y_data[-1] > y_data[0] else "decreasing"
        volatility = np.std(y_data)

        print(f"Overall trend: {trend}")
        print(f"Volatility (standard deviation): {volatility:.2f}")

        if volatility > np.mean(y_data) * 0.5:
            print("• High volatility: values change significantly")
        else:
            print("• Low volatility: relatively stable values")

        print("• Peaks indicate high values")
        print("• Troughs indicate low values")
        print("• Steep slopes indicate rapid changes")

def main():
    """Main function to run the Data Graph Explorer"""
    # Load data
    df = load_csv()

    # Display dataset information
    columns_list = display_data_info(df)

    # Graph creation loop
    while True:
        create_graph(df, columns_list)

        continue_choice = input("\nWould you like to create another graph? (y/n): ").strip().lower()
        if continue_choice not in ['y', 'yes']:
            break

    print("\n" + "="*50)
    print("🎉 Thank you for using Data Graph Explorer!")
    print("="*50)

# Run the main function
if __name__ == "__main__":
    main()